In [ ]:
# Code block generated with Gemini; prompt "All seeds needed for reproducibility in pytorch based DL project"

import os
import random
import numpy as np
import torch

def set_all_seeds(seed_value=42):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.enabled = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_all_seeds(42)

In [ ]:
import cv2
from tqdm import tqdm
import numpy as np

def calculate_blur_score_fft(input_data, radius=30):
    if isinstance(input_data, str):
        img = cv2.imread(input_data, 0)
    else:
        if len(input_data.shape) == 3:
                # Handle float (0-1) vs uint8 (0-255)
                if input_data.dtype != np.uint8:
                    input_data = (input_data * 255).astype(np.uint8)
                img = cv2.cvtColor(input_data, cv2.COLOR_RGB2GRAY)
        else:
            img = input_data
    rows, cols = img.shape
    crow, ccol = rows // 2 , cols // 2
    dft = np.fft.fft2(img)
    dft_shift = np.fft.fftshift(dft) # this function helps shift all the zero freq points to centre
    magnitude_spectrum = np.abs(dft_shift)
    mask = np.zeros((rows, cols), np.uint8)
    cv2.circle(mask, (ccol, crow), radius, 1, -1)
    low_freq_area = magnitude_spectrum * mask
    high_freq_area = magnitude_spectrum * (1 - mask)
    low_energy = np.sum(low_freq_area)
    high_energy = np.sum(high_freq_area)
    ratio = high_energy / low_energy
    return ratio

root_dir = "/home/nam/projects/sid/Motion-Deblurring/gopro_deblur/blur/images"
image_and_blur_score = []

for file in tqdm(os.listdir(root_dir)):
    full_path = os.path.join(root_dir, file)
    blur_score = calculate_blur_score_fft(full_path).item()
    image_and_blur_score.append([full_path, round(blur_score, 3)])

# Sorting in descending order according to blur_score, so that we can then split this into buckets of low, medium and high
# A high ratio means the image is sharp
sorted_blur_scores = sorted(image_and_blur_score, key=lambda x: x[1], reverse=True)

In [ ]:
low_idx = len(sorted_blur_scores)//3
mid_idx = (len(sorted_blur_scores)*2)//3
high_idx = len(sorted_blur_scores)

low_blur_bucket, mid_blur_bucket, high_blur_bucket = sorted_blur_scores[:low_idx], sorted_blur_scores[low_idx:mid_idx], sorted_blur_scores[mid_idx:high_idx]
print(len(low_blur_bucket), len(mid_blur_bucket), len(high_blur_bucket))

In [ ]:
low_blur_sample = random.choice(low_blur_bucket)
mid_blur_sample = random.choice(mid_blur_bucket)
high_blur_sample = random.choice(high_blur_bucket)

### DETR Inference

- Code taken from Github homepage of DETR - colab hands on example. The below codeblock is NOT mine

Link to below code block: https://colab.research.google.com/github/facebookresearch/detr/blob/colab/notebooks/detr_attention.ipynb

In [ ]:
import math

from PIL import Image
import requests
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
from torch import nn
from torchvision.models import resnet50
import torchvision.transforms as T
torch.set_grad_enabled(False)


# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

# standard PyTorch mean-std input image normalization
transform = T.Compose([
    T.Resize(800),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

model = torch.hub.load('facebookresearch/detr', 'detr_resnet50', pretrained=True)
model.eval()

In [ ]:
# for output bounding box post-processing
def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)
    return b

def plot_results(pil_img, prob, boxes, plot_it=True):
    if plot_it:
        plt.figure(figsize=(10,4))
        plt.imshow(pil_img)
        ax = plt.gca()
    colors = COLORS * 100
    for p, (xmin, ymin, xmax, ymax), c in zip(prob, boxes.tolist(), colors):
        if plot_it:
            ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                    fill=False, color=c, linewidth=3))
            cl = p.argmax()
            text = f'{CLASSES[cl]}: {p[cl]:0.2f}'
            ax.text(xmin, ymin, text, fontsize=15,
                    bbox=dict(facecolor='yellow', alpha=0.5))
        else:
            cl = p.argmax()
            text = f'{CLASSES[cl]}: {p[cl]:0.2f}'
            print(text)
    if plot_it:
        plt.axis('off')
        plt.show()

### Inferring the DETR Model on the Blur vs its Sharp counterpart

- This is to get a visual understanding of how the model might behave differently

In [ ]:
def infer_detr_and_plot_with_conf(image_path):
    im = Image.open(image_path)

    # mean-std normalize the input image (batch-size: 1)
    img = transform(im).unsqueeze(0)

    # propagate through the model
    outputs = model(img)

    # keep only predictions with 0.7+ confidence
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probas.max(-1).values > 0.5

    # convert boxes from [0; 1] to image scales
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)
    plot_results(im, probas[keep], bboxes_scaled)

infer_detr_and_plot_with_conf(low_blur_sample[0])
print("***"*50)
infer_detr_and_plot_with_conf(low_blur_sample[0].replace("/blur", "/sharp"))
print("==="*50)
infer_detr_and_plot_with_conf(mid_blur_sample[0])
print("***"*50)
infer_detr_and_plot_with_conf(mid_blur_sample[0].replace("/blur", "/sharp"))
print("==="*50)
infer_detr_and_plot_with_conf(high_blur_sample[0])
print("***"*50)
infer_detr_and_plot_with_conf(high_blur_sample[0].replace("/blur", "/sharp"))
print("==="*50)

In [ ]:
def calculate_iou(box1, box2):
    """Calculates Intersection over Union (IoU) between two boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def get_detr_outputs(image_path, threshold=0.5):
    """Helper to return raw probabilities and scaled boxes."""
    im = Image.open(image_path)
    img = transform(im).unsqueeze(0)
    outputs = model(img)
    
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probas.max(-1).values > threshold
    
    boxes = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)
    probs = probas[keep]
    
    return probs, boxes

def run_comparison(sharp_path, blur_path, iou_threshold=0.5):
    print(f"Comparing:\nSharp: {sharp_path}\nBlur:  {blur_path}\n")
    
    s_probs, s_boxes = get_detr_outputs(sharp_path)
    b_probs, b_boxes = get_detr_outputs(blur_path)
    
    matched_blur_indices = set()
    
    print(f"{'Label':<15} | {'Sharp Conf':<12} | {'Blur Conf':<12} | {'Status'}")
    print("-" * 60)

    for i, s_box in enumerate(s_boxes):
        s_cls = s_probs[i].argmax()
        s_conf = s_probs[i][s_cls].item()
        label = CLASSES[s_cls]
        
        best_iou = 0
        match_idx = -1
        
        for j, b_box in enumerate(b_boxes):
            iou = calculate_iou(s_box.tolist(), b_box.tolist())
            if iou > best_iou:
                best_iou = iou
                match_idx = j
        
        if best_iou > iou_threshold:
            b_cls = b_probs[match_idx].argmax()
            b_conf = b_probs[match_idx][b_cls].item()
            matched_blur_indices.add(match_idx)
            
            diff = s_conf - b_conf
            status = f"Matched (IoU: {best_iou:.2f})"
            if s_cls != b_cls:
                status += f" [CLASS MISMATCH: {CLASSES[b_cls]}]"
            
            print(f"{label:<15} | {s_conf:.4f}     | {b_conf:.4f}     | {status}")
        else:
            print(f"{label:<15} | {s_conf:.4f}     | {'N/A':<12} | LOST IN BLUR")

    # Check for "Ghost" detections (found in blur but not in sharp)
    for j, b_box in enumerate(b_boxes):
        if j not in matched_blur_indices:
            b_cls = b_probs[j].argmax()
            b_conf = b_probs[j][b_cls].item()
            print(f"{CLASSES[b_cls]:<15} | {'N/A':<12} | {b_conf:.4f}     | GHOST DETECTION (Blur Only)")

# Usage Example:
# sample = high_blur_sample[0]
run_comparison(low_blur_sample[0].replace("/blur/images", "/deblur_richardson_filter"), low_blur_sample[0])
run_comparison(mid_blur_sample[0].replace("/blur/images", "/deblur_richardson_filter"), mid_blur_sample[0])
run_comparison(high_blur_sample[0].replace("/blur/images", "/deblur_richardson_filter"), high_blur_sample[0])